<a href="https://colab.research.google.com/github/archakrishnan7/Data-Acquisition-Task-/blob/main/Data_Acquisition_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
# Reading CSV file
population = pd.read_csv('/content/population.csv')
population

,country,population
0,United States,1278642419
1,China,792846414
2,India,1001406378
3,Japan,1206263687
4,Germany,428734972
5,Russia,420968276
6,Brazil,675094950
7,United Kingdom,674991378
8,France,434389014
9,Italy,254467210


In [3]:
# Reading EXCEL file
gdp = pd.read_excel('/content/gdp.xlsx')
gdp

,Country,GDP
0,United States,10450436373455
1,China,5086547998939
2,India,14540718118693
3,Japan,12473835857216
4,Germany,2038442396490
5,Russia,10987125474745
6,Brazil,8793052816619
7,United Kingdom,11799848684255
8,France,7758669752269
9,Italy,12907234326850


In [4]:
# Reading JSON file
internet_users = pd.read_json('/content/internet_users.json')
internet_users

,country,internet_users
0,United States,745595490
1,China,685275917
2,India,606850704
3,Japan,1112451555
4,Germany,246827121
5,Russia,162156967
6,Brazil,24027075
7,United Kingdom,266047041
8,France,216417853
9,Italy,87191493


In [5]:
# Reading XML file
literacy_rate = pd.read_xml('/content/literacy_rate.xml')
literacy_rate

,name,literacy_rate
0,United States,72.7
1,China,82.2
2,India,80.3
3,Japan,97.5
4,Germany,92.9
5,Russia,89.1
6,Brazil,81.0
7,United Kingdom,82.9
8,France,97.6
9,Italy,83.7


In [6]:
import xml.etree.ElementTree as ET
tree = ET.parse('literacy_rate.xml')
root = tree.getroot()

data = []
for country in root.findall('country'):
  data.append({'country': country.get('name'),'literacy_rate':country.find('literacy_rate').text})
literacy = pd.DataFrame(data)
print(literacy)

           country literacy_rate
0    United States          72.7
1            China          82.2
2            India          80.3
3            Japan          97.5
4          Germany          92.9
5           Russia          89.1
6           Brazil          81.0
7   United Kingdom          82.9
8           France          97.6
9            Italy          83.7
10          Canada          70.8
11       Australia          71.6
12     South Korea          66.4
13           Spain          60.6
14          Mexico          76.5
15       Indonesia          75.4
16          Turkey          71.4
17    Saudi Arabia          60.5
18       Argentina          67.8
19    South Africa          87.7


Transform Data

In [7]:
literacy_rate = literacy_rate.rename(columns={'name':'country'})
gdp = gdp.rename(columns={'Country':'country'})

In [8]:
#Standardize country names across all datasets
set(gdp['country']) == set(internet_users['country']) == set(population['country']) == set(literacy_rate['country']) #checking whether name matches if true already countries are standardized.

True

In [ ]:
print(gdp.columns)
print(internet_users.columns)
print(literacy_rate.columns)
print(population.columns)

Index(['country', 'GDP'], dtype='object')
Index(['country', 'internet_users'], dtype='object')
Index(['country', 'literacy_rate'], dtype='object')
Index(['country', 'population'], dtype='object')


In [9]:
#Convert numeric columns to appropriate types
gdp['GDP'] = pd.to_numeric(gdp['GDP'], errors='coerce') # pd.to_numeric used for convert to numeric data type
internet_users['internet_users'] = pd.to_numeric(
    internet_users['internet_users'],
    errors='coerce'
)
literacy_rate['literacy_rate'] = pd.to_numeric(
    literacy_rate['literacy_rate'],
    errors='coerce'
)
population['population'] = pd.to_numeric(
    population['population'],
    errors='coerce'
)

In [10]:
print(gdp.dtypes)
print(internet_users.dtypes)
print(literacy_rate.dtypes)
print(population.dtypes)

country    object
GDP         int64
dtype: object
country           object
internet_users     int64
dtype: object
country           object
literacy_rate    float64
dtype: object
country       object
population     int64
dtype: object


In [11]:
#Handle missing or erroneous data
gdp['GDP'] = gdp['GDP'].fillna(gdp['GDP'].mean())
internet_users['internet_users'] = internet_users['internet_users'].fillna(internet_users['internet_users'].mean())
literacy_rate['literacy_rate'] = literacy_rate['literacy_rate'].fillna(literacy_rate['literacy_rate'].mean())
population['population'] = population['population'].fillna(population['population'].mean())


In [12]:
#Create new columns such as internet penetration rate (internet users / population × 100)
merged_data = gdp.merge(internet_users,on='country')
merged_data = merged_data.merge(literacy_rate,on='country')
merged_data = merged_data.merge(population,on='country')
merged_data['internet_penetration_rate'] = ((merged_data['internet_users']/merged_data['population'])*100).round(2)
print(merged_data[['country','internet_penetration_rate']])

           country  internet_penetration_rate
0    United States                      58.31
1            China                      86.43
2            India                      60.60
3            Japan                      92.22
4          Germany                      57.57
5           Russia                      38.52
6           Brazil                       3.56
7   United Kingdom                      39.41
8           France                      49.82
9            Italy                      34.26
10          Canada                      62.17
11       Australia                      45.90
12     South Korea                      47.17
13           Spain                      88.46
14          Mexico                      20.75
15       Indonesia                       4.91
16          Turkey                      52.14
17    Saudi Arabia                      11.37
18       Argentina                      29.42
19    South Africa                      80.61


In [13]:
merged_data.columns

Index(['country', 'GDP', 'internet_users', 'literacy_rate', 'population',
       'internet_penetration_rate'],
      dtype='object')

Integrate Data

In [14]:
#Merge the datasets based on the country name
merge_data = gdp.merge(internet_users,on='country')
merge_data = merged_data.merge(literacy_rate,on='country')
merge_data = merged_data.merge(population,on='country')
print(merge_data)

           country             GDP  internet_users  literacy_rate  \
0    United States  10450436373455       745595490           72.7   
1            China   5086547998939       685275917           82.2   
2            India  14540718118693       606850704           80.3   
3            Japan  12473835857216      1112451555           97.5   
4          Germany   2038442396490       246827121           92.9   
5           Russia  10987125474745       162156967           89.1   
6           Brazil   8793052816619        24027075           81.0   
7   United Kingdom  11799848684255       266047041           82.9   
8           France   7758669752269       216417853           97.6   
9            Italy  12907234326850        87191493           83.7   
10          Canada   8158351525190       894102645           70.8   
11       Australia   5444267975247       201619113           71.6   
12     South Korea   6562909541266       291770691           66.4   
13           Spain  15275513099842

In [15]:
#Handle unmatched or missing records
print(set(gdp['country'])==set(internet_users['country'])==set(literacy_rate['country'])==set(population['country']))

True


In [16]:
#Validate the data for consistency and completeness

print("Missing values in each column:")
print(merged_data.isnull().sum())
print("\nDuplicate countries:")
print(merged_data['country'].duplicated().sum())
print("\nNumber of unique countries:")
print(merged_data['country'].nunique())
print("\nData types:")
print(merged_data.dtypes)
print("\nCountries with negative GDP:")
print(merged_data[merged_data['GDP'] < 0])
print("\nCountries with negative population:")
print(merged_data[merged_data['population'] < 0])
print("\nCountries with invalid literacy rates:")
print(
    merged_data[
        (merged_data['literacy_rate'] < 0) |
        (merged_data['literacy_rate'] > 100)
    ]
)
print("\nCountries with negative internet users:")
print(merged_data[merged_data['internet_users'] < 0])
if 'internet_penetration_rate' in merged_data.columns:
    print("\nCountries with invalid internet penetration rate:")
    print(
        merged_data[
            (merged_data['internet_penetration_rate'] < 0) |
            (merged_data['internet_penetration_rate'] > 100)
        ]
    )
if (
    merged_data.isnull().sum().sum() == 0
    and merged_data['country'].duplicated().sum() == 0
):
    print("Data validation completed successfully and Data set is consistent and complete")

else:
    print("Data is not consistent and complete")

Missing values in each column:
country                      0
GDP                          0
internet_users               0
literacy_rate                0
population                   0
internet_penetration_rate    0
dtype: int64

Duplicate countries:
0

Number of unique countries:
20

Data types:
country                       object
GDP                            int64
internet_users                 int64
literacy_rate                float64
population                     int64
internet_penetration_rate    float64
dtype: object

Countries with negative GDP:
Empty DataFrame
Columns: [country, GDP, internet_users, literacy_rate, population, internet_penetration_rate]
Index: []

Countries with negative population:
Empty DataFrame
Columns: [country, GDP, internet_users, literacy_rate, population, internet_penetration_rate]
Index: []

Countries with invalid literacy rates:
Empty DataFrame
Columns: [country, GDP, internet_users, literacy_rate, population, internet_penetration_rate]
Index: [

Analyze Data

In [17]:
merged_data.columns

Index(['country', 'GDP', 'internet_users', 'literacy_rate', 'population',
       'internet_penetration_rate'],
      dtype='object')

In [18]:
#Find countries with highest internet penetration rates
top_countries = merged_data.sort_values(by='internet_penetration_rate',ascending=False)
print(top_countries[['country','internet_penetration_rate']])

           country  internet_penetration_rate
3            Japan                      92.22
13           Spain                      88.46
1            China                      86.43
19    South Africa                      80.61
10          Canada                      62.17
2            India                      60.60
0    United States                      58.31
4          Germany                      57.57
16          Turkey                      52.14
8           France                      49.82
12     South Korea                      47.17
11       Australia                      45.90
7   United Kingdom                      39.41
5           Russia                      38.52
9            Italy                      34.26
18       Argentina                      29.42
14          Mexico                      20.75
17    Saudi Arabia                      11.37
15       Indonesia                       4.91
6           Brazil                       3.56


In [19]:
#Calculate average literacy rate across countries
avg_literacy_rate = merged_data['literacy_rate'].mean()
print("Average Literacy Rate:",avg_literacy_rate)


Average Literacy Rate: 78.43


In [20]:
#Investigate relationships between GDP, population, literacy, and internet usage
correlation = merged_data[['GDP','population','internet_users','literacy_rate','internet_penetration_rate']].corr()
print(correlation)

                                GDP  population  internet_users  \
GDP                        1.000000    0.183118        0.212559   
population                 0.183118    1.000000        0.749573   
internet_users             0.212559    0.749573        1.000000   
literacy_rate             -0.081700   -0.229585        0.074808   
internet_penetration_rate  0.002859    0.358561        0.837566   

                           literacy_rate  internet_penetration_rate  
GDP                            -0.081700                   0.002859  
population                     -0.229585                   0.358561  
internet_users                  0.074808                   0.837566  
literacy_rate                   1.000000                   0.235260  
internet_penetration_rate       0.235260                   1.000000  


SQLite3

In [21]:
import sqlite3
#Create an SQLite database
conn =sqlite3.connect("Country.db")


In [22]:
#Load the final merged dataset into a database table
merged_data.to_sql("country_data",conn,if_exists='replace',index = False)



20

In [25]:
#a. Countries with highest internet penetration
query = """select country,internet_penetration_rate
from country_data
order by internet_penetration_rate desc limit 10;"""
result = pd.read_sql(query,conn)
print(result)

         country  internet_penetration_rate
0          Japan                      92.22
1          Spain                      88.46
2          China                      86.43
3   South Africa                      80.61
4         Canada                      62.17
5          India                      60.60
6  United States                      58.31
7        Germany                      57.57
8         Turkey                      52.14
9         France                      49.82


In [26]:
#b. Average literacy rate
query = """select avg(literacy_rate) as average_literacy_rate from country_data"""
result = pd.read_sql(query,conn)
print(result)

   average_literacy_rate
0                  78.43


In [28]:
#c. Correlation between literacy rate and internet penetration
df = pd.read_sql("select literacy_rate,internet_penetration_rate from country_data",conn)
correlation = df["literacy_rate"].corr(df["internet_penetration_rate"])
print("correlation",correlation)

correlation 0.2352598619210605


In [30]:
#d. GDP per capita analysis
query = """select country,GDP/population as GDP_per_capita from country_data
order by GDP_per_capita desc;"""
result = pd.read_sql(query,conn)
print(result)


           country  GDP_per_capita
0        Indonesia          132792
1            Italy           50722
2           Russia           26099
3           France           17861
4   United Kingdom           17481
5            Spain           16998
6            India           14520
7           Brazil           13024
8        Australia           12393
9      South Korea           10609
10          Turkey           10437
11           Japan           10340
12    Saudi Arabia            9239
13   United States            8173
14           China            6415
15       Argentina            6118
16          Canada            5672
17         Germany            4754
18          Mexico            4249
19    South Africa            3630


In [34]:
#e. Which countries have a GDP per capita above $10,000?
query = """ select country,GDP/population as GDP_per_capita from country_data where GDP_per_capita >10000  ;"""
result = pd.read_sql(query,conn)
print(result)

           country  GDP_per_capita
0            India           14520
1            Japan           10340
2           Russia           26099
3           Brazil           13024
4   United Kingdom           17481
5           France           17861
6            Italy           50722
7        Australia           12393
8      South Korea           10609
9            Spain           16998
10       Indonesia          132792
11          Turkey           10437


In [35]:
#f. What is the total population covered in the dataset?
query = """select sum(population) as total_population from country_data;"""
result = pd.read_sql(query,conn)
print(result)

   total_population
0       14864685089


In [37]:
#g. Which countries have the lowest literacy rates, and how does that impact internet access?
query = """select country,literacy_rate,internet_penetration_rate from country_data
order by literacy_rate asc;"""
result = pd.read_sql(query,conn)
print(result)


           country  literacy_rate  internet_penetration_rate
0     Saudi Arabia           60.5                      11.37
1            Spain           60.6                      88.46
2      South Korea           66.4                      47.17
3        Argentina           67.8                      29.42
4           Canada           70.8                      62.17
5           Turkey           71.4                      52.14
6        Australia           71.6                      45.90
7    United States           72.7                      58.31
8        Indonesia           75.4                       4.91
9           Mexico           76.5                      20.75
10           India           80.3                      60.60
11          Brazil           81.0                       3.56
12           China           82.2                      86.43
13  United Kingdom           82.9                      39.41
14           Italy           83.7                      34.26
15    South Africa      

In [38]:
#h. What are the top 5 wealthiest countries by total GDP, and how does that compare with population size?
query = """select country,GDP,population from country_data
order by GDP desc limit 5;"""
result = pd.read_sql(query,conn)
print(result)


     country             GDP  population
0      Spain  15275513099842   898664919
1      India  14540718118693  1001406378
2      Italy  12907234326850   254467210
3      Japan  12473835857216  1206263687
4  Indonesia  12404144955318    93409749


In [39]:
#i.Find countries where internet users exceed 70% of the population.
query = """select country,internet_penetration_rate from country_data where internet_penetration_rate >70;"""
result = pd.read_sql(query,conn)
print(result)

        country  internet_penetration_rate
0         China                      86.43
1         Japan                      92.22
2         Spain                      88.46
3  South Africa                      80.61


In [40]:
#j. What is the average GDP per capita for countries with internet penetration above 50%?
query = """select avg(GDP/population) as average_GDP_per_capita from country_data
where internet_penetration_rate >50;"""
result = pd.read_sql(query,conn)
print(result)


   average_GDP_per_capita
0             8993.222222


In [41]:
#k. How many countries have a literacy rate above 90%, and what is their average internet penetration?
query ="""select count(*) as number_of_countries,avg(internet_penetration_rate)as average_internet_penetration from country_data
where literacy_rate >90;"""
result = pd.read_sql(query,conn)
print(result)


   number_of_countries  average_internet_penetration
0                    3                     66.536667
